# Phase 7: consolidation, certification, and recentered controls

This notebook expands the bounded quadratic-order survey, gives the Q(ζ₅) type (1,5) result an exhaustive interval certificate, and reruns the full six-dimensional compatible-metric search around the new arithmetic records themselves. All comparisons use the polarization-scaled metric and are made only within a fixed type D.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from gkp_systole import (
    CyclotomicFivePolarization, ImaginaryQuadraticOrder, QuadraticHermitianForm,
    SystoleLedgerEntry, certify_cyclotomic_five_systole_interval,
    cyclotomic_five_moduli_family, high_precision_coordinate_systole,
    quadratic_hermitian_moduli_family, reduce_quadratic_hermitian_form,
    refine_compatible_moduli_sample, refine_compatible_moduli_simplex,
    scan_compatible_moduli,
    survey_quadratic_hermitian_polarizations, write_systole_ledger,
)

print('project root:', ROOT)

project root: .


## Exact elementary deduplication

The reducer quotients integral shears, mode exchange, and unit multiplication. It is exact and idempotent, but it is intentionally not advertised as a complete isometry classifier for arbitrary binary Hermitian forms.

In [2]:
order = ImaginaryQuadraticOrder(-4)
reduced = QuadraticHermitianForm(order, 2, 2, 1, 1)
sheared = QuadraticHermitianForm(order, 2, 6, 3, 1)
print('reduced form :', reduce_quadratic_hermitian_form(reduced))
print('sheared form :', sheared)
print('canonicalized:', reduce_quadratic_hermitian_form(sheared))
assert reduce_quadratic_hermitian_form(sheared) == reduced

reduced form : QuadraticHermitianForm(order=ImaginaryQuadraticOrder(discriminant=-4), a=2, c=2, first=1, second=1)
sheared form : QuadraticHermitianForm(order=ImaginaryQuadraticOrder(discriminant=-4), a=2, c=6, first=3, second=1)
canonicalized: QuadraticHermitianForm(order=ImaginaryQuadraticOrder(discriminant=-4), a=2, c=2, first=1, second=1)


## Expanded exact quadratic-order survey

We enlarge Phase 6 from |Δ| ≤ 100 and diagonal bound 12 to |Δ| ≤ 160 and diagonal bound 16. Determinants 2 through 8 are included. This remains a bounded candidate survey, not a classification theorem.

In [3]:
MAX_DISCRIMINANT = 160
MAX_DIAGONAL = 16
discriminants = tuple(-n for n in range(3, MAX_DISCRIMINANT + 1) if (-n) % 4 in (0, 1))
started = perf_counter()
quadratic_surveys = {
    determinant: survey_quadratic_hermitian_polarizations(
        discriminants, determinant, maximum_diagonal=MAX_DIAGONAL
    )
    for determinant in range(2, 9)
}
quadratic_records = {}
for survey in quadratic_surveys.values():
    for item in survey:
        quadratic_records.setdefault(item.form.polarization_type, item)
print('elapsed seconds:', round(perf_counter() - started, 3))
print('deduplicated candidates:', sum(map(len, quadratic_surveys.values())))
print('type      Delta       form (a,c,z1,z2)       exact q/sqrt(|Delta|)        decimal')
for kind, item in sorted(quadratic_records.items()):
    form = item.form
    data = (form.a, form.c, form.first, form.second)
    exact = f'{item.squared_systole_coefficient}/sqrt({form.order.radicand})'
    print(f'{str(kind):9s} {form.order.discriminant:6d} {str(data):>24s} {exact:>26s} {item.squared_systole:12.9f}')

elapsed seconds: 114.979
deduplicated candidates: 6302
type      Delta       form (a,c,z1,z2)       exact q/sqrt(|Delta|)        decimal
(1, 2)        -4             (2, 2, 1, 1)                  2/sqrt(4)  1.000000000
(1, 3)       -24             (6, 6, 3, 2)                 4/sqrt(24)  0.816496581
(1, 4)       -36             (7, 7, 3, 2)               7/2/sqrt(36)  0.583333333
(1, 5)       -55          (15, 15, 2, -4)                 4/sqrt(55)  0.539359890
(1, 6)        -3            (3, 3, 2, -1)                  1/sqrt(3)  0.577350269
(1, 7)       -68           (14, 14, 6, 3)                 4/sqrt(68)  0.485071250
(1, 8)       -47          (12, 12, 7, -3)                 3/sqrt(47)  0.437594974
(2, 2)        -8             (4, 4, 2, 2)                  2/sqrt(8)  0.707106781
(2, 4)        -4             (4, 4, 2, 2)                  1/sqrt(4)  0.500000000


## Exhaustive interval certificate for the simple quartic candidate

The certificate uses an outward-rounded interval period metric. A Gershgorin lower bound limits every lift that could beat the initial representative; all such lifts are then enumerated. This upgrades the earlier 70-digit recheck to a finite exhaustive interval calculation.

In [4]:
quartic = CyclotomicFivePolarization(2, -1)
certificate = certify_cyclotomic_five_systole_interval(quartic, decimal_places=70)
print('certified:', certificate.certified)
print('ell^2 interval:', certificate.lower_bound, certificate.upper_bound)
print('interval width:', certificate.interval_width)
print('multiplicities:', certificate.class_multiplicity, certificate.lift_multiplicity)
print('gap to next candidate:', certificate.separation_gap)
print('recognized expression:', certificate.algebraic_expression)
print('annihilating polynomial coefficients:', certificate.annihilating_polynomial)

certified: True
ell^2 interval: 0.5505527681884694152828838327643550718103597344032634653462703062476354 0.5505527681884694152828838327643550718103597344032634653462703062476396
interval width: 4.238057808890901387994267828135486553359424600729116672724663768489815e-69
multiplicities: 10 10
gap to next candidate: 0.1299678784931625304623485648860537859819613886085900401231218876546568
recognized expression: sqrt(4/25 + 8*sqrt(5)/125)
annihilating polynomial coefficients: (3125, 0, -1000, 0, 16)


## Recentered full-dimensional controls

Each arithmetic candidate is now the origin of its own six-coordinate Sp(4,R)/U(2) chart. We screen 2,500 deterministic Halton points and refine the strongest points. The final random-direction refinement checks combined tangent directions that a coordinate-only pattern search can miss.

In [5]:
winner_13 = quadratic_records[(1, 3)].form
family_13 = quadratic_hermitian_moduli_family(winner_13, name='Delta=-24 type (1,3) record')
family_15 = cyclotomic_five_moduli_family(quartic, name='Q(zeta_5) type (1,5) record')
SEARCH = dict(
    sample_count=2500, radius=1.75, local_fraction=0.35, local_radius=0.25,
    refinement_starts=10, refinement_initial_step=0.18,
    refinement_minimum_step=1e-5, refinement_rounds=120,
)
searches = {}
for offset, family in enumerate((family_13, family_15)):
    started = perf_counter()
    scan = scan_compatible_moduli(family, seed=20260713 + offset, **SEARCH)
    directional = refine_compatible_moduli_sample(
        family, scan.best_sample, seed=705 + offset, direction_count=24,
        initial_step=0.08, minimum_step=2e-5, maximum_rounds=90,
    )
    if family.polarization.type == (1, 5):
        for extra_seed in (707, 708, 709):
            directional = refine_compatible_moduli_sample(
                family, directional, seed=extra_seed, direction_count=64,
                initial_step=0.04, minimum_step=5e-6, maximum_rounds=140,
            )
        directional = refine_compatible_moduli_simplex(
            family, directional, initial_step=0.002,
            coordinate_tolerance=1e-9, value_tolerance=1e-12,
            maximum_iterations=2500,
        )
    searches[family.polarization.type] = (scan, directional)
    checked = high_precision_coordinate_systole(
        family, directional.coordinates, decimal_places=60
    )
    print(f'{family.polarization.type}:')
    print('  arithmetic reference :', family.reference_ell_squared)
    print('  best screened point  :', scan.best_screen_sample.squared_systole)
    print('  best refined control :', directional.squared_systole)
    print('  controls beating ref :', scan.number_beating_reference)
    print('  60-digit recheck     :', checked)
    print('  coordinate norm      :', sum(x*x for x in directional.coordinates)**0.5)
    print('  elapsed seconds      :', round(perf_counter() - started, 3))

(1, 3):
  arithmetic reference : 0.8164965809277261
  best screened point  : 0.7316691834807638
  best refined control : 0.8155721844746697
  controls beating ref : 0
  60-digit recheck     : 0.815572184474670112804002205544646273501753519133451828987931
  coordinate norm      : 0.0033955381071406644
  elapsed seconds      : 15.672


(1, 5):
  arithmetic reference : 0.5505527681884692
  best screened point  : 0.5408423014379317
  best refined control : 0.632455505731361
  controls beating ref : 6
  60-digit recheck     : 0.632455505731336668479612340850546671142652758816846966424818
  coordinate norm      : 2.8981680951193955
  elapsed seconds      : 107.664


## Interpretation

The exact Δ = -24 type (1,3) record remains unbeaten in this recentered chart. For type (1,5), however, the broad compatible-metric search finds a stronger numerical point near ell² = sqrt(2/5), substantially above the Q(ζ₅) CM reference. At the end of Phase 7 this exact identification and the competitor's CM status were unknown. Phase 7.5, in notebook 18, reconstructs the point exactly, proves ell² = sqrt(2/5), and proves that it is another CM surface.

## Machine-readable ledger

Every entry records its normalization and claim strength. In particular, `bounded arithmetic record`, `best observed numerical control`, and `global optimizer` are never conflated.

In [6]:
entries = []
for kind, item in sorted(quadratic_records.items()):
    form = item.form
    entries.append(SystoleLedgerEntry(
        candidate_id=f'quadratic_Delta_{form.order.discriminant}_type_{kind}',
        phase=7, dimension_g=2, polarization_type=str(kind),
        family='square of an imaginary-quadratic CM elliptic curve',
        cm_data=f'Delta={form.order.discriminant}; form={(form.a, form.c, form.first, form.second)}',
        ell_squared_decimal=f'{item.squared_systole:.16g}',
        ell_squared_exact=f'{item.squared_systole_coefficient}/sqrt({form.order.radicand})',
        class_multiplicity=item.core_systole_result.class_multiplicity,
        lift_multiplicity=item.core_systole_result.lift_multiplicity,
        metric_convention='polarization_scaled', arithmetic_status='exact certified CVP',
        search_status='best bounded quadratic-order candidate',
        search_scope=f'|Delta|<={MAX_DISCRIMINANT}; diagonal<={MAX_DIAGONAL}; determinant 2..8',
    ))
quartic_result = quartic.compute_relative_systole()
entries.append(SystoleLedgerEntry(
    candidate_id='zeta5_type_1_5', phase=7, dimension_g=2, polarization_type='(1, 5)',
    family='simple quartic CM', cm_data='Q(zeta_5), Phi={1,2}, alpha=(2,-1)',
    ell_squared_decimal=certificate.upper_bound,
    ell_squared_exact=certificate.algebraic_expression or '',
    class_multiplicity=certificate.class_multiplicity, lift_multiplicity=certificate.lift_multiplicity,
    metric_convention='polarization_scaled', arithmetic_status='exhaustive interval-certified CVP',
    search_status='CM candidate, beaten by numerical compatible-metric control',
    search_scope='Q(zeta_5) coefficient family plus recentered six-dimensional control scan',
))
control_15 = searches[(1, 5)][1]
entries.append(SystoleLedgerEntry(
    candidate_id='type_1_5_best_phase7_control', phase=7, dimension_g=2, polarization_type='(1, 5)',
    family='generic compatible metric control',
    cm_data='CM status not established', ell_squared_decimal=f'{control_15.squared_systole:.16g}',
    ell_squared_exact='', class_multiplicity=control_15.class_multiplicity,
    lift_multiplicity=control_15.lift_multiplicity, metric_convention='polarization_scaled',
    arithmetic_status='60-digit numerical recheck; not interval certified',
    search_status='best observed Phase-7 numerical control',
    search_scope='2500-point six-dimensional Halton scan, multi-direction and simplex refinements',
    notes=f'coordinates={control_15.coordinates}',
))
json_path = ROOT / 'data' / 'phase7_result_ledger.json'
csv_path = ROOT / 'data' / 'phase7_result_ledger.csv'
write_systole_ledger(entries, json_path=json_path, csv_path=csv_path)
print('wrote:', json_path)
print('wrote:', csv_path)
print('entries:', len(entries))

wrote: data/phase7_result_ledger.json
wrote: data/phase7_result_ledger.csv
entries: 11
